In [1]:
import pandas as pd
from sqlalchemy import create_engine

db_user = 'postgres'
db_password = 'postgres'
db_host = 'localhost'
db_port = '5432'
db_name = 'puzzle'

engine = create_engine(f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}')

ModuleNotFoundError: No module named 'psycopg2'

In [ ]:
gas_stations_query = """
SELECT * FROM gas_stations
ORDER BY random()
LIMIT 5;
"""
gas_stations_df = pd.read_sql(gas_stations_query, engine)
gas_stations_df.head()

In [ ]:
gas_stations_df.describe()

In [ ]:
gas_prices_query = """
SELECT * FROM gas_prices
ORDER BY random()
LIMIT 5;
"""
gas_prices_df = pd.read_sql(gas_prices_query, engine)
gas_prices_df.head()

In [ ]:
gas_prices_df.describe()

In [ ]:
# train the model
# Load all data
gas_stations_query = """
SELECT * FROM gas_stations WHERE id = 377;
"""
gas_stations_df = pd.read_sql(gas_stations_query, engine)
gas_stations_df.head()

In [ ]:
gas_prices_query = """
SELECT * FROM gas_prices WHERE gas_station_id = 377;
"""
gas_prices_df = pd.read_sql(gas_prices_query, engine)
print(gas_prices_df.count())
gas_prices_df.head()

In [ ]:
x = gas_prices_df[["gas_station_id", "fuel_type", "created_at"]].copy()
x = pd.get_dummies(x, columns=["fuel_type", "gas_station_id"])

x["created_at"] = pd.to_datetime(x["created_at"])
x["year"] = x["created_at"].dt.year
x["month"] = x["created_at"].dt.month
x["day"] = x["created_at"].dt.day
x = x.drop(columns=["created_at"])

x.head()

In [ ]:
y = gas_prices_df["price"].copy()
y.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

model = LinearRegression()
model.fit(x_train, y_train)

accuracy = model.score(x_test, y_test)
print(f"Model accuracy: {accuracy*100:.2f}%")

In [ ]:
model_features = model.feature_names_in_

new_price = pd.DataFrame({
    "gas_station_id": [377],
    "fuel_type": ['Euro95 E10'],
    "created_at": ["2026-03-20"]
})
new_price = pd.get_dummies(new_price, columns=["fuel_type", "gas_station_id"])

new_price["created_at"] = pd.to_datetime(new_price["created_at"])
new_price["year"] = new_price["created_at"].dt.year
new_price["month"] = new_price["created_at"].dt.month
new_price["day"] = new_price["created_at"].dt.day
new_price = new_price.reindex(columns=model_features, fill_value=False)
new_price.head()

prediction = model.predict(new_price)

print("Prediction:", prediction[0])